In [1]:
import numpy as np
import torch
import torch.nn as nn
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

In [2]:
# PMM Implementation [from PMMs.ipynb]
class ParametricMatrixModel(nn.Module):
    def __init__(self, matrix_size=2):
        super().__init__()
        self.matrix_size = matrix_size
        num_params = (matrix_size * (matrix_size + 1)) // 2  # Symmetric matrix parameters
        
        # Initialize parameters for M0 and M1 matrices
        self.M0 = nn.Parameter(torch.randn(num_params))
        self.M1 = nn.Parameter(torch.randn(num_params))

    def build_matrix(self, params):
        """Construct symmetric matrix from packed parameters"""
        matrix = torch.zeros((self.matrix_size, self.matrix_size))
        idx = 0
        for i in range(self.matrix_size):
            for j in range(i, self.matrix_size):
                matrix[i,j] = matrix[j,i] = params[idx]
                idx += 1
        return matrix

    def forward(self, x):
        # x shape: (batch_size, 1)
        batch_size = x.shape[0]
        
        # Build M0 and M1 matrices
        M0 = self.build_matrix(self.M0)
        M1 = self.build_matrix(self.M1)
        
        # Create batch of matrices: M(x) = M0 + x*M1
        M = M0.unsqueeze(0) + x.view(-1, 1, 1) * M1.unsqueeze(0)
        
        # Compute eigenvalues (ascending order)
        eigenvalues = torch.linalg.eigh(M)[0]
        
        # Return smallest eigenvalue
        return eigenvalues[:, 0]

In [3]:
# Apply PMM to introduce noise to the quantum state
def apply_pmm(state, pmm_model, noise_level):
    """Apply Parametric Matrix Model (PMM) to introduce noise."""
    dim = len(state)
    I = np.eye(dim)
    depolarizing_channel = (1 - noise_level) * I + noise_level / dim * np.ones((dim, dim))
    
    # Here we use the learned model (from PMM) to modify the depolarizing channel
    pmm_output = pmm_model(torch.tensor([noise_level], dtype=torch.float32))
    noisy_state = np.dot(depolarizing_channel, state) * pmm_output.item()  # Applying PMM to noise matrix
    
    return noisy_state

In [4]:
# Define a simple quantum circuit in Numpy
def apply_hadamard(state):
    """Apply Hadamard gate to a 1-qubit state."""
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    return np.dot(H, state)

def apply_cnot(state):
    """Apply CNOT gate to a 2-qubit state."""
    CNOT = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0]
    ])
    return np.dot(CNOT, state)

In [5]:
# Initial state: |00⟩
initial_state = np.array([1, 0, 0, 0])

# Apply Hadamard on qubit 0
state_after_h = apply_hadamard(np.array([1, 0]))  # Hadamard on qubit 0
state_combined = np.kron(state_after_h, np.array([1, 0]))  # Tensor with |0⟩

# Apply CNOT to create a Bell state
final_state_numpy = apply_cnot(state_combined)

# Initialize PMM model
pmm_model = ParametricMatrixModel(matrix_size=2)

# Apply PMM with a small noise level (0.05)
noisy_state_numpy = apply_pmm(final_state_numpy, pmm_model, noise_level=0.05)

# Normalize noisy state
noisy_state_numpy /= np.linalg.norm(noisy_state_numpy)
print("Noisy state using PMM in NumPy:", noisy_state_numpy)

Noisy state using PMM in NumPy: [-0.70687445 -0.01812499 -0.01812499 -0.70687445]


In [6]:
# Qiskit simulation
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Transpile and run using Qiskit Aer
backend = AerSimulator()
transpiled_qc = transpile(qc, backend)

# Add depolarizing noise in Qiskit
noise_model = NoiseModel()

# Add 1-qubit depolarizing error for Hadamard
error_1q = depolarizing_error(0.05, 1) # 5% noise
noise_model.add_all_qubit_quantum_error(error_1q, ['h'])

# Add 2-qubit depolarizing error for CNOT
error_2q = depolarizing_error(0.05, 2) 
noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

# Run with noise model
job_noisy = backend.run(transpiled_qc, shots=1024, noise_model=noise_model)
result_noisy_qiskit = job_noisy.result()
counts_noisy_qiskit = result_noisy_qiskit.get_counts()

print("Noisy Qiskit results with depolarizing noise:", counts_noisy_qiskit)

Noisy Qiskit results with depolarizing noise: {'01': 11, '10': 14, '11': 504, '00': 495}


In [7]:
# Compare Results
def compare_results(state_numpy, counts_qiskit):
    """Compare NumPy state with Qiskit results."""
    probabilities_numpy = np.abs(state_numpy) ** 2
    print("\nProbabilities from NumPy PMM Model:")
    for i, prob in enumerate(probabilities_numpy):
        print(f"State |{format(i, '02b')}>: {prob:.4f}")

    print("\nQiskit Counts with Noise:")
    total_counts = sum(counts_qiskit.values())
    for state, count in counts_qiskit.items():
        prob_qiskit = count / total_counts
        print(f"State |{state}>: {prob_qiskit:.4f}")

compare_results(noisy_state_numpy, counts_noisy_qiskit)


Probabilities from NumPy PMM Model:
State |00>: 0.4997
State |01>: 0.0003
State |10>: 0.0003
State |11>: 0.4997

Qiskit Counts with Noise:
State |01>: 0.0107
State |10>: 0.0137
State |11>: 0.4922
State |00>: 0.4834
